# Mixed Models for Ordinal Data

This notebook demonstrates how to fit **continuation ratio (CR) mixed models** for ordinal
longitudinal data using the `glmmadaptive` Python package.

The CR approach re-expresses the ordinal likelihood as a standard binomial GLMM on expanded
pseudo-observation data, requiring only two helper functions:

- `cr_setup(y)` — expands the ordinal response into binary pseudo-observations
- `cr_marg_probs(eta)` — converts CR linear predictors to marginal category probabilities

Ported from the R vignette `vignettes/Ordinal_Mixed_Models.Rmd`.

## Continuation Ratio Model — Theory

Let $y_{ij}$ take values in $\{0, 1, \ldots, K\}$.  The **forward** CR mixed model:

$$
\log\!\left\{\frac{\Pr(y_{ij}=k\mid y_{ij}\geq k)}{1-\Pr(y_{ij}=k\mid y_{ij}\geq k)}\right\}
= \alpha_k + x_{ij}^\top\beta + z_{ij}^\top b_i
$$

The **backward** formulation conditions on $y_{ij}\leq k$ instead.

Marginal probabilities (forward):
$$\Pr(y=k) = \sigma(\eta_k)\prod_{k'<k}[1-\sigma(\eta_{k'})], \quad
\Pr(y=K)=1-\sum_{k<K}\Pr(y=k)$$

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import expit

from glmmadaptive import MixedModel, MixModResults, cr_setup, cr_marg_probs
from glmmadaptive.families import Binomial

## 1. Simulating Ordinal Longitudinal Data

We simulate $n = 300$ subjects with $K = 8$ measurements each under the **forward** CR model.
The ordinal outcome has 4 levels: **none (0), mild (1), moderate (2), severe (3)**.

In [ ]:
rng = np.random.default_rng(1234)

n     = 300   # number of subjects
K_obs = 8     # measurements per subject
t_max = 15    # maximum follow-up time

ids  = np.repeat(np.arange(n), K_obs)
time = np.concatenate([
    np.concatenate([[0], np.sort(rng.uniform(0, t_max, K_obs - 1))])
    for _ in range(n)
])
sex_subj = rng.choice(["male", "female"], size=n)
sex      = np.repeat(sex_subj, K_obs)

df = pd.DataFrame({"id": ids, "time": time, "sex": sex})

# Fixed-effects design (no intercept)
sex_female = (df["sex"] == "female").astype(float).values
X = np.column_stack([sex_female, df["time"].values, sex_female * df["time"].values])
Z = np.column_stack([np.ones(len(df)), df["time"].values])

# True parameters
thrs  = np.array([-1.5, 0.0, 0.9])
betas = np.array([-0.25, 0.24, -0.05])
D11, D22 = 0.48, 0.10

b = np.column_stack([
    rng.normal(0, np.sqrt(D11), n),
    rng.normal(0, np.sqrt(D22), n),
])

eta_common = X @ betas + (Z * b[ids]).sum(axis=1)
eta_matrix = eta_common[:, None] + thrs[None, :]   # (n_total, 3)
mprobs     = cr_marg_probs(eta_matrix, direction="forward")

y_codes    = np.array([rng.choice(4, p=row) for row in mprobs])
cat_labels = ["none", "mild", "moderate", "severe"]
df["y"]    = pd.Categorical.from_codes(y_codes, categories=cat_labels)

print(df.head(10))
print("\nCategory counts:\n", df["y"].value_counts().sort_index())

## 2. Data Preparation

`cr_setup()` expands each ordinal observation into binary pseudo-observations and creates
the `cohort` factor that encodes which CR comparison each row belongs to.

In [ ]:
cr_vals = cr_setup(df["y"], direction="forward")

cr_data = df.iloc[cr_vals["subs"]].copy().reset_index(drop=True)
cr_data["y_new"]  = cr_vals["y"]
cr_data["cohort"] = cr_vals["cohort"]

print(f"Original rows: {len(df)},  expanded rows: {len(cr_data)}")
print(f"\nCohort distribution:\n{cr_data['cohort'].value_counts()}")

## 3. Basic Continuation Ratio Model

Fit a random-intercepts CR model.  The `cohort` variable captures the threshold parameters $\alpha_k$.

In [ ]:
fm = MixedModel(
    fixed  = "y_new ~ cohort + sex + time",
    random = "~ 1 | id",
    data   = cr_data,
    family = Binomial(),
).fit(verbose=False)

print(fm.summary())

Coefficients have a **log odds ratio** interpretation: `exp(coef)` is the OR for a unit
increase in the covariate, holding the CR conditioning event fixed.

## 4. Relaxing the CR Assumption

We can allow the sex effect to vary by category by adding a `cohort * sex` interaction.

In [ ]:
gm = MixedModel(
    fixed  = "y_new ~ cohort * sex + time",
    random = "~ 1 | id",
    data   = cr_data,
    family = Binomial(),
).fit(verbose=False)

print(gm.summary())

## 5. Likelihood Ratio Test

Use `MixModResults.anova()` to compare the two models.

In [ ]:
lrt = MixModResults.anova(fm, gm)
print(lrt.to_string())

A non-significant p-value indicates that the sex effect satisfies the CR ordinality
assumption (the same across categories).  We proceed with the simpler model `fm`.

## 6. Effect Plots — Conditional CR Probabilities

We visualise the predicted **conditional** CR probabilities (at the population mean, $b_i = 0$)
for each cohort level over time.

In [ ]:
time_grid     = np.linspace(0, 10, 55)
cohort_levels = list(cr_data["cohort"].cat.categories)

grid_rows = []
for c in cohort_levels:
    for s in ["male", "female"]:
        for t in time_grid:
            grid_rows.append({"cohort": c, "sex": s, "time": t, "id": 0})

nDF           = pd.DataFrame(grid_rows)
nDF["cohort"] = pd.Categorical(nDF["cohort"], categories=cohort_levels)
nDF["pred"]   = fm.predict(newdata=nDF, type_pred="mean_subject")

fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
cohort_colors = dict(zip(cohort_levels, ["#1f77b4", "#ff7f0e", "#2ca02c"]))

for ax, sex_val in zip(axes, ["male", "female"]):
    sub = nDF[nDF["sex"] == sex_val]
    for cohort_val, grp in sub.groupby("cohort", observed=True):
        ax.plot(grp["time"], grp["pred"], label=cohort_val,
                color=cohort_colors.get(cohort_val, "grey"), linewidth=2)
    ax.set_title(f"Sex: {sex_val}")
    ax.set_xlabel("Follow-up time")
    ax.set_ylim(0, 1)

axes[0].set_ylabel("Conditional CR probability")
axes[0].legend(title="Cohort")
fig.suptitle("Conditional Continuation Ratio Probabilities", fontsize=13)
plt.tight_layout()
plt.show()

## 7. Effect Plots — Marginal Category Probabilities

Conditional CR probabilities are not directly interpretable as $\Pr(Y = k)$.  We convert
them to **marginal category probabilities** using `cr_marg_probs()`.

We extract the fitted coefficients and build the $\eta$ matrix manually — one column per
CR cohort, one row per grid point.

In [ ]:
coefs = fm.fixef()
print(coefs)

In [ ]:
intercept     = coefs["Intercept"]
d_cohort_mild = coefs.get("cohort[T.y>=mild]",     0.0)
d_cohort_mod  = coefs.get("cohort[T.y>=moderate]", 0.0)
beta_sex_male = coefs.get("sex[T.male]",           0.0)
beta_time     = coefs["time"]

alpha_all  = intercept
alpha_mild = intercept + d_cohort_mild
alpha_mod  = intercept + d_cohort_mod

marg_rows = []
for s in ["male", "female"]:
    for t in time_grid:
        eta_cov = beta_sex_male * (s == "male") + beta_time * t
        eta_vec = np.array([
            alpha_all  + eta_cov,
            alpha_mild + eta_cov,
            alpha_mod  + eta_cov,
        ])
        marg_rows.append({"sex": s, "time": t, "eta": eta_vec})

marg_df         = pd.DataFrame(marg_rows)
eta_matrix_pred = np.stack(marg_df["eta"].values)
mprobs_pred     = cr_marg_probs(eta_matrix_pred, direction="forward")

for k, lbl in enumerate(cat_labels):
    marg_df[lbl] = mprobs_pred[:, k]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
cat_colors = {"none": "#1f77b4", "mild": "#ff7f0e",
              "moderate": "#2ca02c", "severe": "#d62728"}

for ax, sex_val in zip(axes, ["male", "female"]):
    sub = marg_df[marg_df["sex"] == sex_val]
    for lbl in cat_labels:
        ax.plot(sub["time"], sub[lbl], label=lbl,
                color=cat_colors[lbl], linewidth=2)
    ax.set_title(f"Sex: {sex_val}")
    ax.set_xlabel("Follow-up time")
    ax.set_ylim(0, 1)

axes[0].set_ylabel("Marginal probability P(Y = k)")
axes[0].legend(title="Category")
fig.suptitle("Marginal Category Probabilities (at b = 0)", fontsize=13)
plt.tight_layout()
plt.show()

## Notes and Limitations

| Feature | R (`GLMMadaptive`) | Python (`glmmadaptive`) |
|---------|-------------------|------------------------|
| `cr_setup()` forward & backward | ✅ | ✅ |
| `cr_marg_probs()` forward & backward | ✅ | ✅ |
| Standard binomial GLMM fitting | ✅ | ✅ |
| LRT via `anova` | ✅ | ✅ |
| Effect plot — conditional CR probs | ✅ `effectPlotData()` | ✅ `predict()` + matplotlib |
| Effect plot — marginal probs (at b=0) | ✅ | ✅ manual `cr_marg_probs()` |
| Effect plot — marginal over RE | ✅ `effectPlotData(..., marginal=TRUE)` | ❌ not yet implemented |
| Confidence bands on effect plots | ✅ delta method | ❌ not yet implemented |

**Tip:** To marginalise over random effects in Python, simulate
$b^{(s)} \sim \mathcal{N}(0, \hat{D})$ for $s = 1, \ldots, S$,
compute `cr_marg_probs()` for each draw, and average the resulting probabilities.